In [ ]:
from pipeline import build_audiosep, separate_audio
import torch

In [ ]:
device = torch.device('mps' if torch.mps.is_available() else 'cpu')
device

In [19]:
model = build_audiosep(
    config_yaml='config/audiosep_base.yaml', 
    checkpoint_path='checkpoint/audiosep_base_4M_steps.ckpt', 
    device=device
)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.dense.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Loaded AudioSep model from [checkpoint/audiosep_base_4M_steps.ckpt]


In [9]:
audio_file = 'audios/27PR8SIngv8_snip_0.mp3'
text = 'cat sounds'
output_file='audios/separated_audio.wav'

# AudioSep processes the audio at 32 kHz sampling rate  
separate_audio(model, audio_file, text, output_file, device)

Separating audio from [audios/27PR8SIngv8_snip_0.mp3] with textual query: [cat sounds]
Separated audio written to [audios/separated_audio.wav]


# Processing NAYA Data

In [10]:
import shutil
from pathlib import Path
from tqdm import tqdm

In [11]:
# 1. Define your paths
source_root = Path('CATS_DATA/raw')
dest_root = Path('CATS_DATA/processed')

In [12]:
# 2. Ensure destination exists
dest_root.mkdir(parents=True, exist_ok=True)

In [16]:
# 3. Gather only non-augmented files
print("Scanning directory for original .mp3 files...")

# This list comprehension filters out any file where '_aug' is in the name
audio_files = [
    f for f in source_root.rglob('*.mp3') 
    if "_aug" not in f.name.lower()
]

print(f"✅ Found {len(audio_files)} original files.")

Scanning directory for original .mp3 files...
✅ Found 2961 original files.


In [21]:
AUDIO_SEP_TEXT = "The sound of a cat"

Yes, it is definitely better to load them at 32 kHz explicitly.

When you're working with a model like AudioSep that has a fixed internal sampling rate (fs​=32 kHz), loading the audio at that exact rate before it hits the model is the "cleanest" way to work.

Here is why loading at 32 kHz upfront is the professional move:
1. Consistency and Control

If you let the separate_audio function handle the resampling, you are relying on whatever library the author used (likely torchaudio or ffmpeg). By loading it yourself with a library like librosa, you can control the resampling quality (e.g., using a high-quality "sinc" interpolator) to ensure you aren't losing subtle cat purr frequencies.
2. Speed and Memory

If you load a 48 kHz file into memory, your computer is storing ~33% more data than the model actually needs. If you're processing hundreds of files, loading them directly at 32 kHz saves RAM and prevents the CPU from having to do "on-the-fly" conversion during the inference loop.
3. Avoiding Aliasing

Cat sounds—especially high-pitched meows or the sharp "t" in a hiss—contain high-frequency components. According to the Nyquist-Shannon Sampling Theorem:
Max Frequency Capable of Being Represented=2fs​​

At 32 kHz, you can capture sounds up to 16 kHz. Most cat vocalizations fall well below this, but by resampling properly (with a low-pass filter), you ensure that frequencies above 16 kHz don't "fold back" into your audio as digital noise (aliasing).

In [25]:
import json
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
import os
from contextlib import redirect_stdout

In [26]:
log_file_path = "processing_files.jsonl"

In [ ]:
processed = []

for file_path in tqdm(audio_files, desc="Separating Cat Audio", unit="file"):
    relative_path = file_path.relative_to(source_root)
    new_file_path = dest_root / relative_path
    new_file_path.parent.mkdir(parents=True, exist_ok=True)

    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "filename": file_path.name,
        "relative_path": str(relative_path),
    }

    try:
        # This block catches all print statements from AudioSep and throws them away
        with open(os.devnull, 'w') as fnull:
            with redirect_stdout(fnull):
                separate_audio(model, file_path, AUDIO_SEP_TEXT, new_file_path, device)
        
        log_entry["status"] = "success"
        processed.append(file_path)
    except Exception as e:
        log_entry["status"] = "failed"
        log_entry["error"] = str(e)
        # Use tqdm.write so the error message doesn't break the progress bar!
        tqdm.write(f"❌ Error on {file_path.name}: {e}")

    # Write to your .jsonl
    with open(log_file_path, "a") as log_file:
        log_file.write(json.dumps(log_entry) + "\n")

Separating Cat Audio:   6%|▌         | 180/2961 [05:11<6:50:43,  8.86s/file] 